# 1: Imports, Path Setup & Load Model

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import json
import pandas as pd

import io
from PIL import ImageFilter
from keras import layers

# Determine repo root directory (two levels up from notebooks/isabel/)
REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "isabel" else Path.cwd()

# Path to saved Keras model inside the artifacts directory
MODEL_PATH = REPO_ROOT / "artifacts" / "gluten_guard_convnexttiny.keras"

# Directory containing test images categorized in subfolders
DATA_DIR = Path(
    "/home/isabelksommerfeld/code/lrnzgll/gluten-guard/data/"
    "food-101-predict-images-40-classes/food-101-predict-images-40-classes/"
)

# Validate that paths exist
assert MODEL_PATH.exists(), f"Model file not found at: {MODEL_PATH}"
assert DATA_DIR.exists(), f"Data directory not found at: {DATA_DIR}"

# Load trained ConvNeXt Tiny model
print(f"Loading model from: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

I0000 00:00:1789388753.179085  106052 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789388753.978740  106052 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789388761.996112  106052 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading model from: /home/isabelksommerfeld/code/lrnzgll/gluten-guard/artifacts/gluten_guard_convnexttiny.keras


E0000 00:00:1789388766.385661  106052 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded successfully!


/home/isabelksommerfeld/code/lrnzgll/gluten-guard/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 184 variables whereas the saved optimizer has 369 variables. 
  saveable.load_own_variables(store)


# 2: Infer Class Names & Define Prediction Function

In [2]:
# Try to load class names from artifacts JSON if available
CLASSES_PATH = REPO_ROOT / "artifacts" / "class_names.json"

if CLASSES_PATH.exists():
    data = json.loads(CLASSES_PATH.read_text())
    class_names = data["class_names"] if isinstance(data, dict) else data
else:
    # Full list of 101 Food-101 classes in standard alphabetical order
    class_names = [
        'apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare',
        'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito',
        'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake',
        'ceviche', 'cheesecake', 'cheese_plate', 'chicken_curry', 'chicken_quesadilla',
        'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder',
        'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes',
        'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict',
        'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras',
        'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice',
        'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich',
        'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup',
        'hot_dog', 'huevos_rancheros', 'hummus', 'ice_cream', 'lasagna',
        'lobster_bisque', 'lobster_roll_sandwich', 'macaroni_and_cheese', 'macarons', 'miso_soup',
        'mussels', 'nachos', 'omelette', 'onion_rings', 'oysters',
        'pad_thai', 'paella', 'pancakes', 'panna_cotta', 'peking_duck',
        'pho', 'pizza', 'pork_chop', 'poutine', 'prime_rib',
        'pulled_pork_sandwich', 'ramen', 'ravioli', 'red_velvet_cake', 'risotto',
        'samosa', 'sashimi', 'scallops', 'seaweed_salad', 'shrimp_and_grits',
        'spaghetti_bolognese', 'spaghetti_carbonara', 'spring_rolls', 'steak', 'strawberry_shortcake',
        'sushi', 'tacos', 'takoyaki', 'tiramisu', 'tuna_tartare', 'waffles'
    ]

print(f"Loaded {len(class_names)} class names.")


def predict_image_top_k(image_input, k: int = 3): ####### MODIFIED (image_path -> image_input)
    """
    Load and preprocess an image (from Path/str or PIL Image),
    then return top-k predictions across all 101 classes.
    Note: ConvNeXt Tiny model's preprocessing is built into the model, so input images
    remain in standard [0, 255] float32 scale.
    """
    # Check if input is already a PIL Image or a file path
    if isinstance(image_input, Image.Image): ####### MODIFIED
        img = image_input.convert("RGB").resize((288, 288)) ####### MODIFIED
    else:
        img = Image.open(image_input).convert("RGB").resize((288, 288)) ####### MODIFIED

    img_array = np.array(img, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)  # Shape: (1, 288, 288, 3)

    probabilities = model.predict(img_batch, verbose=0)[0]

    # Sort probabilities to get top-k predicted indices
    top_indices = np.argsort(probabilities)[::-1][:k]

    results = [(class_names[idx], probabilities[idx]) for idx in top_indices]
    return img, results

Loaded 101 class names.


# 3: Full Dataset Evaluation Table (Per-Class & Overall Summary)

In [ ]:
# 3: Full Dataset Evaluation Table (Per-Class & Overall Summary)
# Collect ALL test images across all subfolders
all_images = list(DATA_DIR.rglob("*.jpg")) + list(DATA_DIR.rglob("*.png"))
print(f"Evaluating all {len(all_images)} images across all class subfolders...")

results = []

for idx, img_path in enumerate(all_images):
    # Direct folder name as ground truth class (matches model class format)
    true_class = img_path.parent.name

    # Load raw image directly
    raw_img = Image.open(img_path).convert("RGB")

    # Run fast prediction on the raw image
    _, top_preds = predict_image_top_k(raw_img, k=1)
    top1_class, top1_conf = top_preds[0]

    is_correct = (top1_class == true_class)

    results.append({
        "file_name": img_path.name,
        "true_class": true_class,
        "predicted_class": top1_class,
        "correct": is_correct
    })

# Convert to DataFrame
df_results = pd.DataFrame(results)

# Group statistics by class and collect all predicted labels as a list
class_summary = (
    df_results.groupby("true_class")
    .agg(
        total_images=("correct", "count"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
        all_predictions=("predicted_class", list)
    )
    .reset_index()
)

# Sort classes alphabetically
class_summary = class_summary.sort_values(by="true_class", ascending=True)

# Format accuracy as percentage
class_summary["accuracy_pct"] = class_summary["accuracy"].map("{:.1%}".format)

# Calculate overall statistics
total_all = class_summary["total_images"].sum()
correct_all = class_summary["correct_predictions"].sum()
overall_acc = correct_all / total_all if total_all > 0 else 0

# Append overall summary row at the bottom
total_row = pd.DataFrame([{
    "true_class": "--- TOTAL / OVERALL ---",
    "total_images": total_all,
    "correct_predictions": correct_all,
    "accuracy": overall_acc,
    "accuracy_pct": f"{overall_acc:.1%}",
    "all_predictions": "-"
}])

final_summary = pd.concat([class_summary, total_row], ignore_index=True)

Evaluating all 150 images across all class subfolders...


I0000 00:00:1789388953.704537  106116 service.cc:153] XLA service 0x76876088c2d0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789388953.704644  106116 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1789388953.927509  106116 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


,true_class,total_images,correct_predictions,accuracy_pct,all_predictions
0,apple_pie,6,5,83.3%,"[apple_pie, apple_pie, chocolate_mousse, apple..."
1,baby_back_ribs,4,4,100.0%,"[baby_back_ribs, baby_back_ribs, baby_back_rib..."
2,baklava,5,4,80.0%,"[baklava, baklava, baklava, baklava, chocolate..."
3,beef_carpaccio,5,5,100.0%,"[beef_carpaccio, beef_carpaccio, beef_carpacci..."
4,beef_tartare,5,5,100.0%,"[beef_tartare, beef_tartare, beef_tartare, bee..."
5,bruschetta,3,3,100.0%,"[bruschetta, bruschetta, bruschetta]"
6,cannoli,3,3,100.0%,"[cannoli, cannoli, cannoli]"
7,carrot_cake,4,4,100.0%,"[carrot_cake, carrot_cake, carrot_cake, carrot..."
8,ceviche,3,2,66.7%,"[ceviche, paella, ceviche]"
9,cheesecake,4,3,75.0%,"[cheesecake, cheesecake, cheesecake, cheese_pl..."


In [5]:
# Force pandas to show full list contents without truncation
with pd.option_context('display.max_colwidth', None):
    display(final_summary[["true_class", "total_images", "correct_predictions", "accuracy_pct", "all_predictions"]])

,true_class,total_images,correct_predictions,accuracy_pct,all_predictions
0,apple_pie,6,5,83.3%,"[apple_pie, apple_pie, chocolate_mousse, apple_pie, apple_pie, apple_pie]"
1,baby_back_ribs,4,4,100.0%,"[baby_back_ribs, baby_back_ribs, baby_back_ribs, baby_back_ribs]"
2,baklava,5,4,80.0%,"[baklava, baklava, baklava, baklava, chocolate_cake]"
3,beef_carpaccio,5,5,100.0%,"[beef_carpaccio, beef_carpaccio, beef_carpaccio, beef_carpaccio, beef_carpaccio]"
4,beef_tartare,5,5,100.0%,"[beef_tartare, beef_tartare, beef_tartare, beef_tartare, beef_tartare]"
5,bruschetta,3,3,100.0%,"[bruschetta, bruschetta, bruschetta]"
6,cannoli,3,3,100.0%,"[cannoli, cannoli, cannoli]"
7,carrot_cake,4,4,100.0%,"[carrot_cake, carrot_cake, carrot_cake, carrot_cake]"
8,ceviche,3,2,66.7%,"[ceviche, paella, ceviche]"
9,cheesecake,4,3,75.0%,"[cheesecake, cheesecake, cheesecake, cheese_plate]"
